# Part II: Academic Analyzer Framework

In [Part I]() of our tutorial, we explained how to load data from a variety of datasets into SFrame objects and MongoDB collections. While the SFrame objects make it easy to answer questions about how various academic trends change over time (see [Part III]() of the tutorial), it is still challenging to answer more complicated questions using these types of objects. For example, it would hard to calculate how many papers in Nature were written by a second author who is from the University of Washington, or how many papers have first authors who published in PLOS ONE in 2014.
To answer these types of more complicated questions, we developed a code framework which provides easy object-oriented access to academic data stored in MongoDB. Our framework uses several basic object classes, such as Author, Paper, and AuthorCollection, that let us use Python code to answer complicated questions. In the following sections, we will explain each object class and give examples of how to use it.

## 1. The Paper Class

The Paper class is based on paper data from the MAG dataset and the AMinerMAG dataset (see tutorial Part I). The main idea behind this class is to make it easy to fetch data on a specific paper. Given a paper ID, it is possible to construct a paper object using the following code:

In [1]:
from paper import *
p = Paper('75508021')

#We can easily get various basic paper attribute
print "Paper's id %s" % p.paper_id
print "Paper's refrences count %s" %  p.references_count
print "Paper's venue name %s" % p.venue_name
print "Paper's publication year %s" %  p.publish_year
print "Paper's title '%s'" % p.title
print "Paper's keywords list %s" % p.keywords_list 
print "Paper's authors ids list -  %s" % p.author_ids_list # we can also get list of author objects using p.authors_list

/home/michael/anaconda2/lib/python2.7/site-packages/h5py/__init__.py:36: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
2018-05-14 03:54:35,224 [MainThread  ] [DEBUG]  Fetching paper 75508021


Paper's id 75508021
Paper's refrences count 8
Paper's venue name Nature
Paper's publication year 2007
Paper's title 'Cell biology: The checkpoint brake relieved'
Paper's keywords list [u'pharmacology', u'climate change', u'quantum physics', u'systems biology', u'astrophysics', u'cell division', u'immunology', u'cell biology', u'genomics', u'neurobiology', u'palaeobiology', u'ecology', u'evolution', u'molecular biology', u'environmental science', u'marine biology', u'structural biology', u'earth science', u'metabolomics', u'functional genomics', u'medicine', u'proteomics', u'biochemistry', u'cancer', u'cell cycle', u'dna', u'evolutionary biology', u'bioinformatics', u'biotechnology', u'drug discovery', u'developmental biology', u'transcriptomics', u'biology', u'computational biology', u'materials science', u'nature', u'genetics', u'geophysics', u'cell signalling', u'signal transduction', u'medical research', u'neuroscience', u'rna', u'astronomy', u'nanotechnology', u'physics']
Paper's a

We can also get more complicated paper features.

In [2]:
print "Did the paper's last author publised in the venue before? Answer: %s" %  p.did_last_author_publish_in_venue() 
print "Paper's total citations in 2015 %s" %  p.get_total_citations_number_in_year(2015)
print "Paper's max citation number in a specific year %s" %  p.get_max_citations_number_in_year()
print "The number of times paper's authors published in the paper's venue in the past - %s" %  p.total_number_of_times_authors_published_in_venue

2018-05-14 03:54:52,672 [MainThread  ] [DEBUG]  Fetching author 83162F0D


Did the paper's last author publised in the venue before? Answer: True
Paper's total citations in 2015 1.0
Paper's max citation number in a specific year 2.0
The number of times paper's authors published in the paper's venue in the past - 2


If the paper is in the AMinerMAG dataset, additional features are available, such as the paper's abstract and ISSN.

In [3]:
p = Paper('778DE072')
print "Paper's abstract: \n\n %s" % p.abstract
print "Paper's ISSN %s" % p.issn

2018-05-14 03:54:56,049 [MainThread  ] [DEBUG]  Fetching paper 778DE072


Paper's abstract: 

 In spite of various cytogenetic works on suborder Heteroptera, the chromosome organization, function and its evolution in this group is far from being fully understood. Cytologically, the family Rhyparochromidae constitutes a heterogeneous group differing in chromosome numbers. This family possesses XY sex mechanism in the majority of the species with few exceptions. In the present work, multiple banding techniques viz., C-banding, base-specific fluorochromes (DAPI/CMA3) and silver nitrate staining have been used to cytologically characterize the chromosomes of the seed plant pest Elasmolomus (Aphanus) sordidus Fabricius, 1787 having 2n=12=8A+2m+XY. One pair of the autosomes was large while three others were of almost equal size. At diplotene, C-banding technique revealed, that three autosomal bivalents show terminal constitutive heterochromatic bands while one medium sized bivalent was euchromatic. Microchromosomes (m-chromosomes) were positively heteropycnotic. A

The Paper class contains many functions that can be used to extract additional data on each paper. Moreover, the full paper's object data can be accessed using the following line:

In [4]:
p._json_data

{u'Aminer Paper ID': u'55a5ce0d65ce60f99bf5c02d',
 u'Authors List Sorted': [u'7FDB7566', u'80418DAE'],
 u'Authors Number': 2,
 u'Conference ID mapped to venue name': u'',
 u'Field of study list': [u'0660586C', u'039D5C06'],
 u'Field of study list names': [None, u'Bioinformatics'],
 u'Fields of study parent list (L0)': [],
 u'Fields of study parent list (L1)': [u'039D5C06'],
 u'Fields of study parent list (L2)': [],
 u'Fields of study parent list names (L0)': [],
 u'Fields of study parent list names (L1)': [u'Bioinformatics'],
 u'Fields of study parent list names (L2)': [],
 u'Journal ID mapped to venue name': u'0BDFC074',
 u'Keywords List': [u'biomedical research', u'bioinformatics'],
 u'MAG Paper ID': u'778DE072',
 u'Normalized paper title': u'first report on c banding fluorochrome staining and nor location in holocentric chromosomes of elasmolomus aphanus sordidus fabricius 1787 heteroptera rhyparochromidae',
 u'Normalized venue name': u'zookeys',
 u'Original paper title': u'First re

More details on the Paper class can be found at [paper.py]() 

# 2. The Author Class

The Author class is based on author data from the MAG dataset (see tutorial Part I). The dataset contains data based on over 22 million authors. The main idea behind this class is to make it easy to fetch data on a specific author using the author's author id. To obtain an author's *author id* we can use the following code:

In [5]:
from author import Author
import re

l = Author.find_authors_id_by_name('tim bernerslee')
print "Matching Authors ids %s" % l

#we can also us e regex to find matching authors ids
r = re.compile("Tim\s+B.*", re.IGNORECASE)
l = Author.find_authors_id_by_name(r)
print "Matching Authors ids %s" %l[:10]

Matching Authors ids [u'79762927']
Matching Authors ids [u'7FD415E0', u'82479236', u'85145FD9', u'7DB99A96', u'7DB0F51C', u'7EEB86E0', u'7F309657', u'7FBB9CFB', u'29542192', u'849D27DC']


Given an Author ID, it is possible to construct an Author object using the following code:

In [6]:
a = Author(u'79762927')
print "Author's full name: %s" % a.fullname
print "Author's papers number: %s" % a.papers_number
print "Author's journals list between 2000-2010: %s" % a.get_venues_list(VenueType.journal, start_year=2000, end_year=2010)
print "Author's predict gender and name's male probability: %s (%s)" % (a.gender, a.male_probability) # predicting the author gender based on his/her first name
print "Author's last publication year in the dataset: %s" % a.last_publication_year

2018-05-14 03:55:25,168 [MainThread  ] [DEBUG]  Fetching author 79762927


Author's full name: tim bernerslee
Author's papers number: 20
Author's journals list between 2000-2010: [u'062B05D6', u'038E80CE', u'09B77941', u'0ACD8486', u'0BB0DA67', u'0BB0DA67', u'046515A4', u'06CF2E55']
Author's predict gender and name's male probability: Male (0.997655237422)
Author's last publication year in the dataset: 2012


We can use the code to find more complex insights for example who is the author's most common coauthor:

In [7]:
from collections import Counter
coauthors = a.get_coauthors_list(start_year=None, end_year=None)
print "Author's number of coauthors %s" % len(coauthors)
c = Counter(coauthors)
print "Author's most common coauthor's ID - %s (number of joint papers %s)" % c.most_common(1)[0]

Author's number of coauthors 90
Author's most common coauthor's ID - 0E390AA0 (number of joint papers 7)


The author's object data can be accessed using the following line:

In [8]:
a._json_data

{u'Affilation by Year Dict': {2006: [u'massachusetts institute of technology',
   u'massachusetts institute of technology',
   u'massachusetts institute of technology',
   u'massachusetts institute of technology'],
  2007: [u'massachusetts institute of technology'],
  2008: [u'massachusetts institute of technology',
   u'massachusetts institute of technology',
   u'massachusetts institute of technology',
   u'university of southampton'],
  2009: [u'massachusetts institute of technology',
   u'massachusetts institute of technology',
   u'massachusetts institute of technology'],
  2010: [u'massachusetts institute of technology',
   u'massachusetts institute of technology',
   u'international association of machinists and aerospace workers']},
 u'Author ID': u'79762927',
 u'Author name': u'tim bernerslee',
 u'Coauthors by Years Dict': {2006: [u'8153503E',
   u'7628BD51',
   u'7628BD51',
   u'76C81530',
   u'8153503E',
   u'78E7DA33',
   u'7918DADD',
   u'813BFDF7',
   u'0E390AA0',
   u'78

More details on the Author class can be found at [author.py]() 

# 3. The Papers Collection Class

The goal of the Papers Collection class is to make it easy to analyze a list of paper objects. The class makes it possible to easily filter papers which were published in a specific year, and to obtain various insights regarding the papers and their authors. For example, let's select papers that were published in Nature journal and are part of the AMinerMAG dataset.

In [9]:
from configs import *
sf = tc.load_sframe(EXTENDED_PAPERS_SFRAME)
sf = sf[sf['Original venue name'] == 'Nature'] # Another option to get paper ids is to use PAPERS_FETCHER.get_papers_ids_by_issn(issn) 
sf = sf[sf['Ref Number'] >= 5]
sf.materialize()
paper_ids = list(sf['Paper ID'])

Now let's define a papers collection object which contains all these papers:

In [10]:
from papers_collection_analyer import *
pc = PapersCollection(paper_ids) # this is a lazy object
#Create a list of papers object
print "Number of retrieved paper ids with at least 5 refs- %s" % len(pc.papers_list)    

2018-05-14 04:20:01,819 [MainThread  ] [DEBUG]  Fetching paper 75CA889C
2018-05-14 04:20:01,827 [MainThread  ] [DEBUG]  Fetching paper 761F4E45
2018-05-14 04:20:01,832 [MainThread  ] [DEBUG]  Fetching paper 76AB3267
2018-05-14 04:20:01,835 [MainThread  ] [DEBUG]  Fetching paper 6001790F
2018-05-14 04:20:01,838 [MainThread  ] [DEBUG]  Fetching paper 768B07EA
2018-05-14 04:20:01,843 [MainThread  ] [DEBUG]  Fetching paper 6198EAF7
2018-05-14 04:20:01,846 [MainThread  ] [DEBUG]  Fetching paper 7E1E6D10
2018-05-14 04:20:01,849 [MainThread  ] [DEBUG]  Fetching paper 75BC4430
2018-05-14 04:20:01,852 [MainThread  ] [DEBUG]  Fetching paper 75508021
2018-05-14 04:20:01,856 [MainThread  ] [DEBUG]  Fetching paper 7779326C
2018-05-14 04:20:01,859 [MainThread  ] [DEBUG]  Fetching paper 780EB796
2018-05-14 04:20:01,862 [MainThread  ] [DEBUG]  Fetching paper 75EF94A8
2018-05-14 04:20:01,865 [MainThread  ] [DEBUG]  Fetching paper 7EBD05D1
2018-05-14 04:20:01,868 [MainThread  ] [DEBUG]  Fetching paper 7

2018-05-14 04:20:02,184 [MainThread  ] [DEBUG]  Fetching paper 772A729E
2018-05-14 04:20:02,187 [MainThread  ] [DEBUG]  Fetching paper 77BEAE07
2018-05-14 04:20:02,191 [MainThread  ] [DEBUG]  Fetching paper 7D913D05
2018-05-14 04:20:02,193 [MainThread  ] [DEBUG]  Fetching paper 61A4DD22
2018-05-14 04:20:02,196 [MainThread  ] [DEBUG]  Fetching paper 61990C71
2018-05-14 04:20:02,200 [MainThread  ] [DEBUG]  Fetching paper 77C1DD7C
2018-05-14 04:20:02,201 [MainThread  ] [DEBUG]  Fetching paper 7528E6E7
2018-05-14 04:20:02,205 [MainThread  ] [DEBUG]  Fetching paper 772A1743
2018-05-14 04:20:02,208 [MainThread  ] [DEBUG]  Fetching paper 767234E8
2018-05-14 04:20:02,211 [MainThread  ] [DEBUG]  Fetching paper 757D19A9
2018-05-14 04:20:02,215 [MainThread  ] [DEBUG]  Fetching paper 7768609C
2018-05-14 04:20:02,217 [MainThread  ] [DEBUG]  Fetching paper 755D71A4
2018-05-14 04:20:02,220 [MainThread  ] [DEBUG]  Fetching paper 7728008B
2018-05-14 04:20:02,223 [MainThread  ] [DEBUG]  Fetching paper 7

2018-05-14 04:20:02,521 [MainThread  ] [DEBUG]  Fetching paper 757C0271
2018-05-14 04:20:02,523 [MainThread  ] [DEBUG]  Fetching paper 75CD012D
2018-05-14 04:20:02,526 [MainThread  ] [DEBUG]  Fetching paper 777661F4
2018-05-14 04:20:02,529 [MainThread  ] [DEBUG]  Fetching paper 77A73A8C
2018-05-14 04:20:02,532 [MainThread  ] [DEBUG]  Fetching paper 7778FE7E
2018-05-14 04:20:02,535 [MainThread  ] [DEBUG]  Fetching paper 771E27C2
2018-05-14 04:20:02,538 [MainThread  ] [DEBUG]  Fetching paper 7769F9C4
2018-05-14 04:20:02,541 [MainThread  ] [DEBUG]  Fetching paper 755C1F1C
2018-05-14 04:20:02,545 [MainThread  ] [DEBUG]  Fetching paper 5AA64787
2018-05-14 04:20:02,548 [MainThread  ] [DEBUG]  Fetching paper 77740C09
2018-05-14 04:20:02,551 [MainThread  ] [DEBUG]  Fetching paper 75A49774
2018-05-14 04:20:02,554 [MainThread  ] [DEBUG]  Fetching paper 75D542F9
2018-05-14 04:20:02,556 [MainThread  ] [DEBUG]  Fetching paper 76999B43
2018-05-14 04:20:02,559 [MainThread  ] [DEBUG]  Fetching paper 7

2018-05-14 04:20:02,853 [MainThread  ] [DEBUG]  Fetching paper 7F7BCEE4
2018-05-14 04:20:02,855 [MainThread  ] [DEBUG]  Fetching paper 5D3E33B1
2018-05-14 04:20:02,858 [MainThread  ] [DEBUG]  Fetching paper 7B2D2C81
2018-05-14 04:20:02,861 [MainThread  ] [DEBUG]  Fetching paper 7888346A
2018-05-14 04:20:02,864 [MainThread  ] [DEBUG]  Fetching paper 7AB8232B
2018-05-14 04:20:02,868 [MainThread  ] [DEBUG]  Fetching paper 7991CBB2
2018-05-14 04:20:02,871 [MainThread  ] [DEBUG]  Fetching paper 7B623A67
2018-05-14 04:20:02,873 [MainThread  ] [DEBUG]  Fetching paper 7A797E4B
2018-05-14 04:20:02,877 [MainThread  ] [DEBUG]  Fetching paper 7B680EA1
2018-05-14 04:20:02,880 [MainThread  ] [DEBUG]  Fetching paper 79BBF8FF
2018-05-14 04:20:02,883 [MainThread  ] [DEBUG]  Fetching paper 79A45D74
2018-05-14 04:20:02,886 [MainThread  ] [DEBUG]  Fetching paper 789FCA19
2018-05-14 04:20:02,889 [MainThread  ] [DEBUG]  Fetching paper 79356550
2018-05-14 04:20:02,892 [MainThread  ] [DEBUG]  Fetching paper 7

2018-05-14 04:20:03,181 [MainThread  ] [DEBUG]  Fetching paper 78D5DFEA
2018-05-14 04:20:03,184 [MainThread  ] [DEBUG]  Fetching paper 7A2B4C64
2018-05-14 04:20:03,187 [MainThread  ] [DEBUG]  Fetching paper 806B0997
2018-05-14 04:20:03,189 [MainThread  ] [DEBUG]  Fetching paper 5C318986
2018-05-14 04:20:03,191 [MainThread  ] [DEBUG]  Fetching paper 789BD504
2018-05-14 04:20:03,194 [MainThread  ] [DEBUG]  Fetching paper 7AA98D8E
2018-05-14 04:20:03,198 [MainThread  ] [DEBUG]  Fetching paper 79A1AB6F
2018-05-14 04:20:03,201 [MainThread  ] [DEBUG]  Fetching paper 7AD36FF5
2018-05-14 04:20:03,203 [MainThread  ] [DEBUG]  Fetching paper 5B5D306D
2018-05-14 04:20:03,206 [MainThread  ] [DEBUG]  Fetching paper 7A779672
2018-05-14 04:20:03,209 [MainThread  ] [DEBUG]  Fetching paper 7B176246
2018-05-14 04:20:03,212 [MainThread  ] [DEBUG]  Fetching paper 7B57312C
2018-05-14 04:20:03,215 [MainThread  ] [DEBUG]  Fetching paper 78E9BD37
2018-05-14 04:20:03,218 [MainThread  ] [DEBUG]  Fetching paper 7

...

2018-05-14 04:21:41,565 [MainThread  ] [DEBUG]  Fetching paper 76CC1474
2018-05-14 04:21:41,569 [MainThread  ] [DEBUG]  Fetching paper 613DF11B
2018-05-14 04:21:41,572 [MainThread  ] [DEBUG]  Fetching paper 7795CEBD
2018-05-14 04:21:41,576 [MainThread  ] [DEBUG]  Fetching paper 76092688
2018-05-14 04:21:41,580 [MainThread  ] [DEBUG]  Fetching paper 764EEF06
2018-05-14 04:21:41,583 [MainThread  ] [DEBUG]  Fetching paper 7E7C54ED
2018-05-14 04:21:41,586 [MainThread  ] [DEBUG]  Fetching paper 7D1B8571
2018-05-14 04:21:41,589 [MainThread  ] [DEBUG]  Fetching paper 75B7C7D5
2018-05-14 04:21:41,592 [MainThread  ] [DEBUG]  Fetching paper 764000FD
2018-05-14 04:21:41,595 [MainThread  ] [DEBUG]  Fetching paper 604B3BB1
2018-05-14 04:21:41,598 [MainThread  ] [DEBUG]  Fetching paper 7680D119
2018-05-14 04:21:41,602 [MainThread  ] [DEBUG]  Fetching paper 7595613C
2018-05-14 04:21:41,605 [MainThread  ] [DEBUG]  Fetching paper 771567E2
2018-05-14 04:21:41,608 [MainThread  ] [DEBUG]  Fetching paper 7

2018-05-14 04:21:41,926 [MainThread  ] [DEBUG]  Fetching paper 784A8F4B
2018-05-14 04:21:41,928 [MainThread  ] [DEBUG]  Fetching paper 7A837C00
2018-05-14 04:21:41,931 [MainThread  ] [DEBUG]  Fetching paper 7B6925E1
2018-05-14 04:21:41,934 [MainThread  ] [DEBUG]  Fetching paper 7B673CEC
2018-05-14 04:21:41,937 [MainThread  ] [DEBUG]  Fetching paper 6303896F
2018-05-14 04:21:41,940 [MainThread  ] [DEBUG]  Fetching paper 7A207418
2018-05-14 04:21:41,943 [MainThread  ] [DEBUG]  Fetching paper 79CA48EE
2018-05-14 04:21:41,946 [MainThread  ] [DEBUG]  Fetching paper 7B5A3626
2018-05-14 04:21:41,949 [MainThread  ] [DEBUG]  Fetching paper 788C88A6
2018-05-14 04:21:41,953 [MainThread  ] [DEBUG]  Fetching paper 7A09AC58
2018-05-14 04:21:41,956 [MainThread  ] [DEBUG]  Fetching paper 7AEDDE5F
2018-05-14 04:21:41,960 [MainThread  ] [DEBUG]  Fetching paper 7AD6F38E
2018-05-14 04:21:41,963 [MainThread  ] [DEBUG]  Fetching paper 78B18699
2018-05-14 04:21:41,966 [MainThread  ] [DEBUG]  Fetching paper 7

2018-05-14 04:21:42,290 [MainThread  ] [DEBUG]  Fetching paper 78E96CB5
2018-05-14 04:21:42,293 [MainThread  ] [DEBUG]  Fetching paper 7AF3541C
2018-05-14 04:21:42,296 [MainThread  ] [DEBUG]  Fetching paper 8076BC62
2018-05-14 04:21:42,299 [MainThread  ] [DEBUG]  Fetching paper 7A7FD58B
2018-05-14 04:21:42,303 [MainThread  ] [DEBUG]  Fetching paper 78D0D159
2018-05-14 04:21:42,306 [MainThread  ] [DEBUG]  Fetching paper 795417CB
2018-05-14 04:21:42,308 [MainThread  ] [DEBUG]  Fetching paper 7AF9242F
2018-05-14 04:21:42,311 [MainThread  ] [DEBUG]  Fetching paper 78683FB7
2018-05-14 04:21:42,315 [MainThread  ] [DEBUG]  Fetching paper 626B8EA2
2018-05-14 04:21:42,318 [MainThread  ] [DEBUG]  Fetching paper 7A02EEE9
2018-05-14 04:21:42,321 [MainThread  ] [DEBUG]  Fetching paper 788CBAF5
2018-05-14 04:21:42,323 [MainThread  ] [DEBUG]  Fetching paper 7AA70AFF
2018-05-14 04:21:42,327 [MainThread  ] [DEBUG]  Fetching paper 7B29CC4E
2018-05-14 04:21:42,330 [MainThread  ] [DEBUG]  Fetching paper 7

2018-05-14 04:21:42,643 [MainThread  ] [DEBUG]  Fetching paper 7CC89FC0
2018-05-14 04:21:42,646 [MainThread  ] [DEBUG]  Fetching paper 8140DE29
2018-05-14 04:21:42,649 [MainThread  ] [DEBUG]  Fetching paper 7BB967EF
2018-05-14 04:21:42,652 [MainThread  ] [DEBUG]  Fetching paper 7C2826C6
2018-05-14 04:21:42,655 [MainThread  ] [DEBUG]  Fetching paper 7C97EEAA
2018-05-14 04:21:42,657 [MainThread  ] [DEBUG]  Fetching paper 7B84A350
2018-05-14 04:21:42,660 [MainThread  ] [DEBUG]  Fetching paper 7CDA097E
2018-05-14 04:21:42,663 [MainThread  ] [DEBUG]  Fetching paper 7BCA7262
2018-05-14 04:21:42,667 [MainThread  ] [DEBUG]  Fetching paper 7CB2C8C8
2018-05-14 04:21:42,670 [MainThread  ] [DEBUG]  Fetching paper 7BE62469
2018-05-14 04:21:42,673 [MainThread  ] [DEBUG]  Fetching paper 7BB6B89F
2018-05-14 04:21:42,676 [MainThread  ] [DEBUG]  Fetching paper 7B9FC302
2018-05-14 04:21:42,679 [MainThread  ] [DEBUG]  Fetching paper 7CAE95E3
2018-05-14 04:21:42,683 [MainThread  ] [DEBUG]  Fetching paper 7

Number of retrevied paper ids with at least 5 refs- 32536


Now we can use the papers collection object to gain various insights regarding the papers in the collection.

In [11]:
print "Min publication year %s" % pc.min_publication_year
print "Max publication year %s" % pc.max_publication_year
print "Last Authors Median Academic Age in 2000 - %s" % pc.last_authors_median_age(2000)
print "Last Authors Median Academic Age in 2010 - %s" % pc.last_authors_median_age(2010)

2018-05-14 04:24:09,424 [MainThread  ] [DEBUG]  Fetching author 7F182519
2018-05-14 04:24:09,431 [MainThread  ] [DEBUG]  Fetching author 2761E14D
2018-05-14 04:24:09,433 [MainThread  ] [DEBUG]  Fetching author 7D63AB2B
2018-05-14 04:24:09,435 [MainThread  ] [DEBUG]  Fetching author 7FBF5B13
2018-05-14 04:24:09,438 [MainThread  ] [DEBUG]  Fetching author 827671A3
2018-05-14 04:24:09,441 [MainThread  ] [DEBUG]  Fetching author 825EC656
2018-05-14 04:24:09,444 [MainThread  ] [DEBUG]  Fetching author 7524B017
2018-05-14 04:24:09,446 [MainThread  ] [DEBUG]  Fetching author 854BF47B
2018-05-14 04:24:09,448 [MainThread  ] [DEBUG]  Fetching author 8077B97A
2018-05-14 04:24:09,451 [MainThread  ] [DEBUG]  Fetching author 8140D565
2018-05-14 04:24:09,453 [MainThread  ] [DEBUG]  Fetching author 7687744E
2018-05-14 04:24:09,456 [MainThread  ] [DEBUG]  Fetching author 7BA06B09
2018-05-14 04:24:09,458 [MainThread  ] [DEBUG]  Fetching author 80C3D5D8
2018-05-14 04:24:09,461 [MainThread  ] [DEBUG]  Fet

Min publication year 1930
Max publication year 2015


2018-05-14 04:24:09,485 [MainThread  ] [DEBUG]  Fetching author 812DB2D8
2018-05-14 04:24:09,487 [MainThread  ] [DEBUG]  Fetching author 7B1681AF
2018-05-14 04:24:09,489 [MainThread  ] [DEBUG]  Fetching author 7D12C98C
2018-05-14 04:24:09,491 [MainThread  ] [DEBUG]  Fetching author 7AA367DE
2018-05-14 04:24:09,494 [MainThread  ] [DEBUG]  Fetching author 8639FDA6
2018-05-14 04:24:09,496 [MainThread  ] [DEBUG]  Fetching author 7B402767
2018-05-14 04:24:09,498 [MainThread  ] [DEBUG]  Fetching author 71724258
2018-05-14 04:24:09,501 [MainThread  ] [DEBUG]  Fetching author 8028D8F6
2018-05-14 04:24:09,503 [MainThread  ] [DEBUG]  Fetching author 7DBB89CA
2018-05-14 04:24:09,506 [MainThread  ] [DEBUG]  Fetching author 7700FBF9
2018-05-14 04:24:09,509 [MainThread  ] [DEBUG]  Fetching author 7CE78A37
2018-05-14 04:24:09,511 [MainThread  ] [DEBUG]  Fetching author 7035CD0E
2018-05-14 04:24:09,514 [MainThread  ] [DEBUG]  Fetching author 7FEDCD6C
2018-05-14 04:24:09,517 [MainThread  ] [DEBUG]  Fet

2018-05-14 04:24:09,782 [MainThread  ] [DEBUG]  Fetching author 7DA7936B
2018-05-14 04:24:09,784 [MainThread  ] [DEBUG]  Fetching author 2B907F51
2018-05-14 04:24:09,787 [MainThread  ] [DEBUG]  Fetching author 81C4F49B
2018-05-14 04:24:09,790 [MainThread  ] [DEBUG]  Fetching author 7D59B23C
2018-05-14 04:24:09,793 [MainThread  ] [DEBUG]  Fetching author 7A6D5EDB
2018-05-14 04:24:09,796 [MainThread  ] [DEBUG]  Fetching author 77FF68D2
2018-05-14 04:24:09,799 [MainThread  ] [DEBUG]  Fetching author 7AAD018F
2018-05-14 04:24:09,801 [MainThread  ] [DEBUG]  Fetching author 7B765D9D
2018-05-14 04:24:09,804 [MainThread  ] [DEBUG]  Fetching author 7D26D18B
2018-05-14 04:24:09,808 [MainThread  ] [DEBUG]  Fetching author 7ECA9814
2018-05-14 04:24:09,810 [MainThread  ] [DEBUG]  Fetching author 7B5D6E32
2018-05-14 04:24:09,813 [MainThread  ] [DEBUG]  Fetching author 81784ABC
2018-05-14 04:24:09,815 [MainThread  ] [DEBUG]  Fetching author 790969DD
2018-05-14 04:24:09,818 [MainThread  ] [DEBUG]  Fet

2018-05-14 04:24:10,127 [MainThread  ] [DEBUG]  Fetching author 77AA1BF2
2018-05-14 04:24:10,135 [MainThread  ] [DEBUG]  Fetching author 75C2E44A
2018-05-14 04:24:10,138 [MainThread  ] [DEBUG]  Fetching author 0E083258
2018-05-14 04:24:10,141 [MainThread  ] [DEBUG]  Fetching author 0AC8A6ED
2018-05-14 04:24:10,144 [MainThread  ] [DEBUG]  Fetching author 7552A99A
2018-05-14 04:24:10,147 [MainThread  ] [DEBUG]  Fetching author 151491E0
2018-05-14 04:24:10,150 [MainThread  ] [DEBUG]  Fetching author 7D810F5F
2018-05-14 04:24:10,154 [MainThread  ] [DEBUG]  Fetching author 7D3A959F
2018-05-14 04:24:10,157 [MainThread  ] [DEBUG]  Fetching author 7DD94926
2018-05-14 04:24:10,159 [MainThread  ] [DEBUG]  Fetching author 7E3E4065
2018-05-14 04:24:10,162 [MainThread  ] [DEBUG]  Fetching author 816E3421
2018-05-14 04:24:10,165 [MainThread  ] [DEBUG]  Fetching author 75A0C6B3
2018-05-14 04:24:10,168 [MainThread  ] [DEBUG]  Fetching author 80ABDB05
2018-05-14 04:24:10,173 [MainThread  ] [DEBUG]  Fet

...

2018-05-14 04:24:14,819 [MainThread  ] [DEBUG]  Fetching author 7FCB7F5F
2018-05-14 04:24:14,822 [MainThread  ] [DEBUG]  Fetching author 7C26F524
2018-05-14 04:24:14,825 [MainThread  ] [DEBUG]  Fetching author 7E097DC3
2018-05-14 04:24:14,828 [MainThread  ] [DEBUG]  Fetching author 803935B0
2018-05-14 04:24:14,830 [MainThread  ] [DEBUG]  Fetching author 812FC7F6
2018-05-14 04:24:14,833 [MainThread  ] [DEBUG]  Fetching author 70FBFCD4
2018-05-14 04:24:14,836 [MainThread  ] [DEBUG]  Fetching author 7D0A5696
2018-05-14 04:24:14,839 [MainThread  ] [DEBUG]  Fetching author 510A2490
2018-05-14 04:24:14,843 [MainThread  ] [DEBUG]  Fetching author 8118FF38
2018-05-14 04:24:14,846 [MainThread  ] [DEBUG]  Fetching author 6F5859FA
2018-05-14 04:24:14,848 [MainThread  ] [DEBUG]  Fetching author 853AC092
2018-05-14 04:24:14,853 [MainThread  ] [DEBUG]  Fetching author 819D7375
2018-05-14 04:24:14,855 [MainThread  ] [DEBUG]  Fetching author 804F2884
2018-05-14 04:24:14,858 [MainThread  ] [DEBUG]  Fet

2018-05-14 04:24:15,147 [MainThread  ] [DEBUG]  Fetching author 1F97894C
2018-05-14 04:24:15,150 [MainThread  ] [DEBUG]  Fetching author 80D3ADF8
2018-05-14 04:24:15,153 [MainThread  ] [DEBUG]  Fetching author 8075DCBE
2018-05-14 04:24:15,156 [MainThread  ] [DEBUG]  Fetching author 78535423
2018-05-14 04:24:15,159 [MainThread  ] [DEBUG]  Fetching author 0106216B
2018-05-14 04:24:15,162 [MainThread  ] [DEBUG]  Fetching author 82B5B6ED
2018-05-14 04:24:15,165 [MainThread  ] [DEBUG]  Fetching author 862855DE
2018-05-14 04:24:15,167 [MainThread  ] [DEBUG]  Fetching author 82275BC0
2018-05-14 04:24:15,170 [MainThread  ] [DEBUG]  Fetching author 7FDC4A3D
2018-05-14 04:24:15,172 [MainThread  ] [DEBUG]  Fetching author 7FEFBC33
2018-05-14 04:24:15,175 [MainThread  ] [DEBUG]  Fetching author 7EF261EC
2018-05-14 04:24:15,178 [MainThread  ] [DEBUG]  Fetching author 6B45776C
2018-05-14 04:24:15,180 [MainThread  ] [DEBUG]  Fetching author 77E973CD
2018-05-14 04:24:15,183 [MainThread  ] [DEBUG]  Fet

2018-05-14 04:24:16,512 [MainThread  ] [DEBUG]  Fetching author 7A0496F4
2018-05-14 04:24:16,515 [MainThread  ] [DEBUG]  Fetching author 8327F1E0
2018-05-14 04:24:16,517 [MainThread  ] [DEBUG]  Fetching author 7A76A6E3
2018-05-14 04:24:16,520 [MainThread  ] [DEBUG]  Fetching author 168AD9D3
2018-05-14 04:24:16,523 [MainThread  ] [DEBUG]  Fetching author 0575D338
2018-05-14 04:24:16,525 [MainThread  ] [DEBUG]  Fetching author 7D862899
2018-05-14 04:24:16,528 [MainThread  ] [DEBUG]  Fetching author 05451FD1
2018-05-14 04:24:16,531 [MainThread  ] [DEBUG]  Fetching author 7FD14AA7
2018-05-14 04:24:16,534 [MainThread  ] [DEBUG]  Fetching author 83C81562
2018-05-14 04:24:16,536 [MainThread  ] [DEBUG]  Fetching author 7DEB1F8F
2018-05-14 04:24:16,539 [MainThread  ] [DEBUG]  Fetching author 834FE7DB
2018-05-14 04:24:16,541 [MainThread  ] [DEBUG]  Fetching author 759D8F18
2018-05-14 04:24:16,545 [MainThread  ] [DEBUG]  Fetching author 79565BCE
2018-05-14 04:24:16,547 [MainThread  ] [DEBUG]  Fet

2018-05-14 04:24:16,836 [MainThread  ] [DEBUG]  Fetching author 75C1ABA5
2018-05-14 04:24:16,839 [MainThread  ] [DEBUG]  Fetching author 0EE30C7F
2018-05-14 04:24:16,843 [MainThread  ] [DEBUG]  Fetching author 7F1D9596
2018-05-14 04:24:16,845 [MainThread  ] [DEBUG]  Fetching author 7EF57CA6
2018-05-14 04:24:16,847 [MainThread  ] [DEBUG]  Fetching author 7E12B7DF
2018-05-14 04:24:16,850 [MainThread  ] [DEBUG]  Fetching author 7F1FCC5E
2018-05-14 04:24:16,852 [MainThread  ] [DEBUG]  Fetching author 7EA59B93
2018-05-14 04:24:16,856 [MainThread  ] [DEBUG]  Fetching author 78FAA12F
2018-05-14 04:24:16,859 [MainThread  ] [DEBUG]  Fetching author 78093263
2018-05-14 04:24:16,862 [MainThread  ] [DEBUG]  Fetching author 85FC2B8B
2018-05-14 04:24:16,865 [MainThread  ] [DEBUG]  Fetching author 7F24974D
2018-05-14 04:24:16,868 [MainThread  ] [DEBUG]  Fetching author 7DB7288D
2018-05-14 04:24:16,871 [MainThread  ] [DEBUG]  Fetching author 7CF867BD
2018-05-14 04:24:16,874 [MainThread  ] [DEBUG]  Fet

Last Authors Median Academic Age in 2010 - 18.449626865671643


Let's calculate what is the most-cited paper in the collection:

In [12]:
p = pc.max_citations_paper(2015, include_self_citations=True)
print p.title, p.total_citations_number_by_year(2015, include_self_citation=True)

Cleavage of Structural Proteins during the Assembly of the Head of Bacteriophage T4 118108.0


Let's calculate the median citation number for papers that were published in 2000 after 5 years:

In [13]:
print "Median citation number for papers that were published in 2000 after 5 years - %s" %  pc.papers_median_citations_after_years(2009, 5, True)

Median citation number for papers that were published in 2000 after 5 years - 77.0


We can also get the papers’ top keywords in various years:

In [14]:
print "Top 10-keywords in 1950 %s" % pc.top_keywords(1980, top_keywords_number=10)
print "Top 10-keywords in 2015 %s" % pc.top_keywords(2015, top_keywords_number=10)

Top 10-keywords in 1950 {u'amino acid': 22, u'genetics': 50, u'biochemistry': 20, u'nucleotide sequence': 20, u'enzyme': 40, u'molecular weight': 25, u'central nervous system': 19, u'dopamine': 18, u'escherichia coli': 20, u'dna sequence': 24}
Top 10-keywords in 2015 {u'genetics': 13, u'cell biology': 13, u'cancer': 15, u'nature': 235, u'rna': 12, u'immunology': 11, u'geochemistry': 15, u'x ray crystallography': 37, u'palaeontology': 16, u'neuroscience': 13}


The Papers Collection class provides wide functionality to obtain insights about the papers that easily can be extended. More details on the Papers Collection class can be found in papers_collections_analyzer.py

# 4. The Authors Collection Class

The goal of the Authors Collection class is to provide an easy way to analyze a variety of features of list of Author objects, such as age and gender statistics. Let's, for example, take all the authors that published in Nature in 2010:

In [15]:
from authors_list_analyzer import AuthorsListAnalyzer
authors_list = pc.all_authors_in_year_list(2010) # We can also consturct the 
ac = AuthorsListAnalyzer(authors_list)
print "Authors average academic age in 2010 - %s" % ac.get_average_age(2010)
print 'Authors average number of publications between 2005 and 2010 - %s' % ac.get_average_publication_number(2005,2010)
print "Authors Gender stats - %s" % ac.get_gender_stats()

Authors average academic age in 2010 - 12.468946188340807
Authors average number of publications between 2005 and 2010 - 19.47690582959641
Authors Gender stats - Counter({u'Male': 4339, u'Unisex': 1954, u'Female': 1611, None: 1016})


# 5. The Venue Class

The Venue class is an extension of the Papers Collection class, and it supports all the Papers Collection class functionality. Moreover, the main goal of the Venue class is to easily analyze venues (especially journals) and to help to understand how various venue's features change over time. The class can be constructed using MAG's venue ID, venue name, ISSN list, or a list of MAG paper IDs. For example, if we want to analyze Science journal, we can use the following line:

In [19]:
from venue import *
# this will search and load all the MAG papers from Science  
v = Venue(venue_name="Science")
# Total papers: 212,305 papers
print "Total paper %s" % len(v.papers_list)
print "Top 10-keywords in 1950 %s" % v.top_keywords(1960, top_keywords_number=10)
print "Top 10-keywords in 2015 %s" % v.top_keywords(2015, top_keywords_number=10)\

2018-05-14 05:05:52,575 [MainThread  ] [INFO ]  Consturcting a Venue object with the following params venue_id=None, venue_name=Science, issn_list=() 
2018-05-14 05:05:52,581 [MainThread  ] [INFO ]  Get SJR data of venue_name=Science, issn_list=()
2018-05-14 05:05:52,587 [MainThread  ] [INFO ]  Consturcted a Venue object with 212305 papers


Total paper 212305
Top 10-keywords in 1950 {u'organic chemistry': 6, u'elementary particles': 6, u'genetics': 14, u'social science': 6, u'genes': 7, u'microbiology': 9, u'social change': 7, u'enzyme': 17, u'amino acid': 6, u'mutants': 6}
Top 10-keywords in 2015 {u'membrane protein': 5, u'human factors': 5, u'bioinformatics': 5, u'injury prevention': 5, u'biomedical research': 5, u'ergonomics': 5, u'biological sciences': 6, u'physical sciences': 7, u'suicide prevention': 5, u'occupational safety': 5}


In this study, we mainly focus on papers with at least 5 references. Therefore, we will use the *papers_filter_func* to filter out the papers without 5 references.

In [25]:
v = Venue(venue_name="Science", papers_filter_func=lambda p:p.references_count < 5)
print "Median number of citations after five years for papers published in 2000 - %s" % v.papers_median_citations_after_years(2000, 5, True)
print "Papers' average length in 2015 - %s" % v.papers_average_length(2015)[0]
print "Last authors average age in 2014 - %s' " % v.last_authors_average_age(2014)


2018-05-14 05:07:31,418 [MainThread  ] [INFO ]  Consturcting a Venue object with the following params venue_id=None, venue_name=Science, issn_list=() 
2018-05-14 05:07:36,604 [MainThread  ] [INFO ]  Get SJR data of venue_name=Science, issn_list=()
2018-05-14 05:07:36,607 [MainThread  ] [INFO ]  Consturcted a Venue object with 212305 papers


Median number of citations after five years for papers publish in 2000 - 93.0
Papers' average length in 2015 - 4.027027027027027
Last authors average age in 2014 - 19.81500872600349' 


It is also possible to construct a venue object using a list of papers IDs. The VenueFetcher class (in venue_fetcher.py) contains a function that provides an easy way to get all papers IDs for various journals:

In [26]:
VENUE_FETCHER.get_valid_venues_papers_ids_sframe(min_ref_number=5, min_journal_papers_num=100)

Journal ID mapped tovenue name ...,Count,Journal name,Paper IDs List
0143CDDC,1041,studies in history andphilosophy of science ...,"[757EC648, 768D63B9,774104E4, 7691EF0A, ..."
0970A6F5,3907,internationalimmunopharmacology ...,"[7722D02E, 75BAFE6A,75F4E11E, 7E4F7951, ..."
04EAD463,177,international journal ofrheumatology ...,"[5B57FF96, 63DE6A9E,80C97B52, 77163112, ..."
0650CB25,164,informal logic,"[7DF2C40E, 7FABBBF2,5E733FCB, 604681B7, ..."
0A976928,185,irish veterinary journal,"[7D954D12, 779BB59F,8152A4BF, 7BEF9B7B, ..."
0AA5FF5A,411,frontiers inneuroendocrinology ...,"[75567664, 7823E030,77E5C2AC, 754D4816, ..."
09C674F8,6906,acs applied materials &interfaces ...,"[75ACBB73, 772F0B15,7771BD64, 7E17C60B, ..."
044AB264,122,bulletin of the americanmuseum of natural his ...,"[7DE92DE1, 7A41CED8,7C1FC428, 7B468860, ..."
038BDE8E,536,oceanology,"[77F1B853, 7D6DFE13,76A71664, 78C77D6D, ..."
0655038B,11681,cancer letters,"[77F08DEE, 759AD03E,7DF1C344, 767EF695, ..."


One of the main usages of the Venue class is to analyze how various features have changed over time. Namely, the features_dict property will return a dict, which includes the venue information and how various features have changed.

In [28]:
v.features_dict

{'end_year': 2016,
 'features': {'authors_average_age': {'authors_average_age': {1918: 0.0,
    1923: 0.0,
    1926: 0.0,
    1928: 0.0,
    1929: 0.0,
    1932: 0.0,
    1934: 0.0,
    1940: 0.0,
    1943: 0.0,
    1946: 0.0,
    1951: 0.0,
    1952: 0.0,
    1954: 0.0,
    1955: 0.0,
    1957: 1.1428571428571428,
    1962: 1.8571428571428572,
    1964: 0.5,
    1965: 0.0,
    1966: 0.95,
    1967: 1.6898734177215189,
    1968: 2.739130434782609,
    1969: 2.3157894736842106,
    1970: 2.659090909090909,
    1971: 2.295238095238095,
    1972: 3.5217391304347827,
    1973: 2.6956521739130435,
    1974: 1.5,
    1975: 2.2828054298642533,
    1976: 2.925925925925926,
    1977: 3.4070796460176993,
    1979: 1.8333333333333333,
    1980: 4.521739130434782,
    1981: 4.0,
    1982: 4.228571428571429,
    1983: 9.75,
    1984: 3.933333333333333,
    1985: 2.8666666666666667,
    1986: 5.078651685393258,
    1987: 5.563218390804598,
    1988: 5.538461538461538,
    1989: 6.0,
    1990: 8.7134

# 6. The Field of Study Class

The FieldOfStudy class is an extension of the Papers Collection class, and it supports all the Papers Collection class functionality. Moreover, the main goal of this class is to easily analyze the fields of study features over time to help understand how various field's features change over time. The class can be constructed using MAG's field of study ID. For example, if we want to analyze the "social network" field, we can use the following line.

In [29]:
# We us FieldsOfStudyFetcher (fetchers.fields_of_study,py) to get the field_id of social networks fields
import re
d = FIELDS_OF_STUDY_FETCHER.get_field_ids_by_name(re.compile(".*social.*network.*", re.IGNORECASE))
print d

{'06D662E0': 'Social network analysis', '05242AA7': 'Social network'}


If we have two fields that contain the words "social network," we will choose to analyze the field “social network analysis” with field ID of 06D662E0. We can use the PapersCollection function to calculate various field of study features. For example, we can find the most-cited articles after five years that are published in the field in each year.

In [31]:
from field_of_study import *
fs = FieldOfStudy(field_id='06D662E0')
fs.get_yearly_most_cited_papers_sframe(citation_after_year=5, max_publish_year=2015)

citation_number,ids,title,venue_name,venue_type,year
9.0,740909C8,Understanding the policylandscape for climate ...,,,2015
36.0,5952A950,What's different aboutsocial media networks? a ...,Management InformationSystems Quarterly ...,journal,2014
68.0,816DE0D4,Value network analysisand value conversion of ...,Journal of IntellectualCapital ...,journal,2013
176.0,803E5C6D,A Review of FacebookResearch in the Social ...,Perspectives onPsychological Science ...,journal,2012
83.0,7A42E9EE,A realist evaluation ofthe role of communities ...,Implementation Science,journal,2011
165.0,7F4AA6EA,A matrix factorizationtechnique with trust ...,conference on recommendersystems ...,,2010
494.0,75F0BB33,Network Analysis in theSocial Sciences ...,Science,journal,2009
267.0,7DC68258,Dynamic Spread ofHappiness in a Large ...,BMJ,journal,2008
465.0,79FFC3D5,Why we twitter:understanding ...,knowledge discovery anddata mining ...,,2007
90.0,7ECDA02D,A Graph-theoreticperspective on centra ...,Social Networks,journal,2006


If we select only papers with at least five references, we can calculate various field of study features using the features_dict
function:

In [33]:
fs = FieldOfStudy(field_id='06D662E0', papers_filter_func=lambda p: p.references_count < 5) # to calculate authors features only papers with at least five references need to be selected
fs.features_dict(add_field_features_over_time=True)

{'end_year': 2016,
 'features': {'authors_average_age': {'authors_average_age': {1975: 1.3333333333333333,
    1979: 1.0,
    1981: 2.6,
    1982: 0.0,
    1984: 3.0,
    1985: 6.5,
    1988: 0.0,
    1989: 0.0,
    1990: 3.0,
    1991: 3.4,
    1992: 0.0,
    1993: 2.5,
    1995: 8.384615384615385,
    1996: 8.5,
    1997: 4.0,
    1998: 5.0,
    1999: 5.956521739130435,
    2000: 4.476190476190476,
    2001: 3.8181818181818183,
    2002: 6.155555555555556,
    2003: 6.407766990291262,
    2004: 5.17,
    2005: 5.497607655502392,
    2006: 6.339449541284404,
    2007: 6.807272727272728,
    2008: 6.848484848484849,
    2009: 7.265326633165829,
    2010: 7.140204271123491,
    2011: 7.368378158109209,
    2012: 7.353016688061618,
    2013: 7.092161016949152,
    2014: 6.979591836734694,
    2015: 7.326855123674911,
    2016: 6.296296296296297}},
  'authors_median_age': {'authors_median_age': {1975: 2.0,
    1979: 1.0,
    1981: 3.0,
    1982: 0.0,
    1984: 3.0,
    1985: 6.5,
    1988